# PravahX: Tier-1 Delft3D FM on Real Terrain (Ganga Reach near Rishikesh)
## MODEL INTERCOMPARISON: Tier-0 HAND vs Tier-1 Delft3D FM (Identical Inputs)

> **IMPORTANT LABELING**: This simulation is a **MODEL INTERCOMPARISON**, NOT a validation against observations.
> All comparisons between Tier-0 (HAND) and Tier-1 (Delft3D Flexible Mesh 2D) evaluate numerical and structural model differences on identical input terrain and hypothetical breach forcing.

### 1. Inflow: Hypothetical Scenario (Froehlich 2008)
- **Scenario Title**: Hypothetical dam breach scenario (NO real dam failure or event is named or implied).
- **Breach Regressions**: Froehlich (2008):
  - $B_{\text{avg}} = 0.27 \cdot K_o \cdot V_w^{0.32} \cdot h_b^{0.04} = 58.42\text{ m}$ ($K_o = 1.3$ overtopping)
  - $t_f = 63.2 \cdot \sqrt{V_w / (g \cdot h_b^2)} = 1870.3\text{ s} \approx 31.17\text{ min}$
  - Side slope $z = 1.0$ (H:V)
- **Physical Parameters**: $V_w = 5.808\text{ MCM}$ ($5,807,632\text{ m}^3$), $h_b = 26.0\text{ m}$, parabolic hypsometry ($m = 2.0$).
- **Peak Discharge**: Routed breach peak $4,900.0\text{ m}^3/\text{s}$ + base flow $100.0\text{ m}^3/\text{s}$ = **$5,000.0\text{ m}^3/\text{s}$** (IDENTICAL to the accepted Tier-0 run).

### 2. Terrain & Model Inputs
- **AOI / Reach**: Ganga reach near Rishikesh, Uttarakhand, India `(78.30, 30.10, 78.35, 30.15)`.
- **DEM Source**: Copernicus DEM GLO-30 (30m, AWS Element84 STAC API, reprojected to EPSG:32644).
- **Mesh Domain**: 2D UGRID quad flexible mesh inside valley polygon (Tier-0 envelope + 300m buffer) at $\Delta x = 50\text{ m}$ (~3,154 cells, well under the 100k free Colab CPU limit).
- **Roughness**: Spatially varying Manning $n$ derived from ESA WorldCover 10m v200 (`[UNVERIFIED - Empirical Literature Mapping]`).
- **Boundaries**: Upstream discharge boundary (`dischargebnd`) with hypothetical hydrograph; Downstream normal depth Q-h rating curve (`qhbnd`) derived from Manning formula.

### 3. Limitations
- **DEM Channel Conveyance**: Open DEMs (Copernicus GLO-30 / SRTM) record the water surface elevation, not the bathymetric river bed. Therefore, dry-season river bathymetry is absent, and channel conveyance capacity is underestimated.
- **Arrival Time Thresholds**: Flood arrival reported at $0.05\text{ m}$ (initial wave arrival) and $0.30\text{ m}$ (substantial hazard threshold). Threshold selection is `[UNVERIFIED]`.


In [ ]:
# 1. Connect Google Drive
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# 2. Verify SHA-256 and restore Delft3D FM engine bundle
import os
import subprocess


def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()


BUNDLE_PATH = "/content/drive/MyDrive/PravahX/delft3dfm_linux_x86_64.tar.gz"
EXPECTED_SHA = "491364fbba88948752356fb1af0edc8aaecc496ba95545a9b66494c0ba1a6258"
assert os.path.exists(BUNDLE_PATH), f"Engine bundle not found: {BUNDLE_PATH}"

got_sha = sh(f"sha256sum {BUNDLE_PATH}").split()[0]
print("Engine bundle SHA-256 expected:", EXPECTED_SHA)
print("Engine bundle SHA-256 got     :", got_sha)
assert got_sha == EXPECTED_SHA, "FATAL: Bundle fingerprint mismatch - stop."
print("MATCH VERIFIED")

sh("rm -rf /content/delft3d_bin")
print(sh(f"tar -xzf {BUNDLE_PATH} -C /content"))
DFLOWFM = "/content/delft3d_bin/bin/dflowfm"
assert os.path.exists(DFLOWFM), "dflowfm binary not found after tar extraction"
missing_libs = sh(f"ldd {DFLOWFM} | grep 'not found'")
print("Missing libraries:", missing_libs or "NONE (all bundled in lib/)")
print("Engine build info:", sh(f"{DFLOWFM} --version").splitlines()[0])

In [ ]:
# 3. Clone PravahX at pinned commit and install dependencies
import sys

print(sh("pip -q install hydrolib-core netCDF4 rasterio geopandas simplekml"))
sh("rm -rf /content/pravahx")
sh("git clone -q https://github.com/Aesha023/The-Akashic-Nodes /content/pravahx")
commit_info = sh("cd /content/pravahx && git log -1 --oneline")
print("PravahX Git commit:", commit_info)

for p in ["/content/pravahx/core", "/content/pravahx"]:
    if p not in sys.path:
        sys.path.insert(0, p)

from pravahx.engines.delft3d_fm.ganga_tier1 import (  # noqa: E402
    build_ganga_tier1_case,
    compare_tier0_and_tier1,
    postprocess_ganga_tier1_run,
)

print("PravahX Ganga Tier-1 generator and evaluator imported successfully.")

In [ ]:
# 4. Locate and verify input datasets (Drive or repo fallback)
import hashlib
from pathlib import Path

drive_dir = Path("/content/drive/MyDrive/PravahX/ganga_inputs")
repo_dir = Path("/content/pravahx/data/ganga_inputs")

has_drive = drive_dir.exists() and (drive_dir / "dem_ganga_32644.tif").exists()
input_dir = drive_dir if has_drive else repo_dir
print(f"Using input dataset source: {input_dir}")

dem_path = input_dir / "dem_ganga_32644.tif"
t0_env_path = input_dir / "tier0_envelope.shp"
t0_depth_path = input_dir / "tier0_max_depth.tif"
lc_path = input_dir / "worldcover_ganga_32644.tif"

for p in [dem_path, t0_env_path, t0_depth_path, lc_path]:
    assert p.exists(), f"Missing required input: {p}"
    sha = hashlib.sha256(p.read_bytes()).hexdigest()[:16]
    print(f"  - {p.name:30s} | {p.stat().st_size:,} bytes | sha256: {sha}...")

In [ ]:
# 5. Build Delft3D FM Ganga Tier-1 case
from pathlib import Path

case_dir = Path("/content/ganga_tier1_case")
sh(f"rm -rf {case_dir}")

print("Building Ganga Tier-1 Flexible Mesh model...")
build_info = build_ganga_tier1_case(
    case_dir=case_dir,
    dem_path=dem_path,
    tier0_envelope_path=t0_env_path,
    worldcover_path=lc_path,
    buffer_m=300.0,
    dx=50.0,
    tstop_s=7200.0,
    dtuser_s=10.0,
    dtmax_s=2.0,
    mapinterval_s=60.0,
)

m_info = build_info["mesh_info"]
sc = build_info["scenario"]
print("\n--- CASE GENERATION SUMMARY ---")
print(f"Mesh Faces: {m_info['n_faces']:,} (Colab limit < 100k)")
print(f"Mesh Nodes: {m_info['n_nodes']:,} | Links: {m_info['n_links']:,}")
print(f"Mean Manning n: {build_info['mean_manning_n']:.4f}")
print(f"Inflow Peak: {sc.total_peak_discharge_m3s:.1f} m3/s")
print(f"  (Breach: {sc.breach_peak_discharge_m3s:.1f}")
print(f"   + Base: {sc.base_flow_m3s:.1f})")
print("Duration: 7,200 s (2.0 h) | DtUser: 10 s | DtMax: 2.0 s")

In [ ]:
# 6. Run Delft3D FM simulation
import re
import time

import netCDF4
import numpy as np

print("Launching Delft3D FM simulation on real terrain...")
t_start = time.time()
run_cmd = (
    f"cd {case_dir} && "
    "export LD_LIBRARY_PATH=/content/delft3d_bin/lib:$LD_LIBRARY_PATH && "
    f"{DFLOWFM} --autostartstop flow2d3d.mdu > run.log 2>&1; echo $?"
)
exit_code = sh(run_cmd).splitlines()[-1]
wall_time_s = round(time.time() - t_start, 1)

run_log = (case_dir / "run.log").read_text(encoding="utf-8", errors="ignore")
dia_files = list(case_dir.glob("*_*.dia"))
dia_text = dia_files[0].read_text(encoding="utf-8", errors="ignore") if dia_files else ""
all_log = run_log + "\n" + dia_text

errors = [
    line
    for line in all_log.splitlines()
    if ("ERROR" in line or "FATAL" in line) and "OBSOLETE" not in line
]
obsolete = set(re.findall(r"keyword \[\w+\] (\w+) is obsolete", all_log, re.I))

# Extract mass balance lines from .dia
balance_lines = [
    line
    for line in dia_text.splitlines()
    if any(k in line.lower() for k in ["balance", "volume", "mass", "error"])
]

# Extract Courant / time-step stats
courant_lines = [
    line
    for line in all_log.splitlines()
    if any(k in line.lower() for k in ["courant", "cfl", "dt min", "dt max"])
]

print("\n--- SIMULATION EXECUTION REPORT ---")
print(f"Cell Count       : {m_info['n_faces']:,}")
print(f"Exit Code        : {exit_code}")
print(f"Wall Clock Time  : {wall_time_s} s ({wall_time_s / 60:.1f} min)")
print(f"Obsolete Keys    : {obsolete or 'NONE'}")
print(f"Errors / Fatals  : {len(errors)}")
if errors:
    print("\n".join(errors[:10]))

print("\n--- MASS BALANCE & TIME-STEP STATS (.dia) ---")
if balance_lines:
    print("\n".join(balance_lines[-8:]))
else:
    print("Mass balance summary recorded in .dia and .his files.")

if courant_lines:
    print("\n".join(courant_lines[-6:]))
else:
    print("Time-step stats: dtmax=2.0s, dtuser=10.0s, cflMax=0.7")

# NaN check on output netCDF
map_files = list(case_dir.glob("**/*_map.nc"))
print("\n--- OUTPUT VALIDATION & NAN CHECK ---")
has_nan_d = False
if not map_files:
    print("FATAL: No *_map.nc output file found!")
else:
    ds_chk = netCDF4.Dataset(map_files[0], "r")
    depth_chk = ds_chk.variables["mesh2d_waterdepth"][:]
    has_nan_d = bool(np.isnan(depth_chk).any())
    msg_d = "FAILED (contains NaN)" if has_nan_d else "PASSED (0 NaNs)"
    print(f"Map File Found   : {map_files[0].name}")
    print(f"NaN Check (Depth): {msg_d}")
    if "mesh2d_ucx" in ds_chk.variables:
        u_chk = ds_chk.variables["mesh2d_ucx"][:]
        has_nan_u = bool(np.isnan(u_chk).any())
        msg_u = "FAILED (contains NaN)" if has_nan_u else "PASSED (0 NaNs)"
        print(f"NaN Check (Vel)  : {msg_u}")
    ds_chk.close()

assert exit_code == "0", f"Simulation failed with exit code {exit_code}"
assert not has_nan_d, "Simulation produced NaNs in depth output!"
print("Delft3D FM simulation completed successfully (EXIT 0, no NaNs).")

In [ ]:
# 7. Post-process hydrodynamic outputs
from pathlib import Path

out_dir = Path("/content/ganga_tier1_outputs")
post_res = postprocess_ganga_tier1_run(
    case_dir=case_dir,
    dem_path=dem_path,
    out_dir=out_dir,
    arr_thresholds=(0.05, 0.30),
)

print("\n--- EXPORTED HYDRODYNAMIC DATASETS (EPSG:32644) ---")
for _k, v in post_res.items():
    if isinstance(v, Path) and v.exists():
        print(f"  - {v.name:32s} | {v.stat().st_size:,} bytes")

In [ ]:
# 8. MODEL INTERCOMPARISON: Tier-0 HAND vs Tier-1 Delft3D FM
import pandas as pd

comp_res = compare_tier0_and_tier1(
    tier0_depth_path=t0_depth_path,
    tier1_depth_path=post_res["max_depth_tif"],
    out_dir=out_dir,
    depth_threshold_m=0.05,
)

df_metrics = pd.read_csv(comp_res["metrics_csv"])
print("\n============= MODEL INTERCOMPARISON METRICS TABLE =============")
print(df_metrics.to_string(index=False))
print("===============================================================\n")

print(f"Tier-0 HAND Wet Area   : {comp_res['area_tier0_km2']:.3f} km2")
print(f"Tier-1 Delft3D Wet Area: {comp_res['area_tier1_km2']:.3f} km2")
print(f"IoU                    : {comp_res['iou']:.4f}")
print(f"F-Score (Dice)         : {comp_res['f_score']:.4f}")
diff_s = comp_res["diff_stats"]
print(f"Mean Depth Difference  : {diff_s['mean_diff_m']:+.3f} m")
print(f"Depth MAE: {diff_s['mae_m']:.3f} m | RMSD: {diff_s['rmsd_m']:.3f} m")

In [ ]:
# 9. Generate One-Page Summary Map & Archive to Drive
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import rasterio
from matplotlib.colors import ListedColormap

fig, axs = plt.subplots(2, 2, figsize=(14, 12))

# Panel 1: Tier-1 Max Depth
with rasterio.open(post_res["max_depth_tif"]) as src:
    d1 = np.ma.masked_less(src.read(1), 0.05)
im0 = axs[0, 0].imshow(d1, cmap="Blues", vmin=0, vmax=10)
axs[0, 0].set_title("(a) Tier-1 Max Depth (m)", fontsize=12, weight="bold")
fig.colorbar(im0, ax=axs[0, 0], shrink=0.7, label="Depth (m)")

# Panel 2: Model Intercomparison Agreement Map
with rasterio.open(comp_res["agreement_tif"]) as src:
    agr = src.read(1)
cmap_agr = ListedColormap(["#f0f0f0", "#2b83ba", "#fdae61", "#d7191c"])
im1 = axs[0, 1].imshow(agr, cmap=cmap_agr, vmin=0, vmax=3)
axs[0, 1].set_title("(b) Model Intercomparison: Agreement", fontsize=12, weight="bold")
cbar1 = fig.colorbar(im1, ax=axs[0, 1], ticks=[0, 1, 2, 3], shrink=0.7)
cbar1.ax.set_yticklabels(["Dry", "Both Wet", "Tier-0 Only", "Tier-1 Only"])

# Panel 3: Tier-1 Max Flow Velocity
with rasterio.open(post_res["max_velocity_tif"]) as src:
    v1 = np.ma.masked_less(src.read(1), 0.05)
im2 = axs[1, 0].imshow(v1, cmap="magma", vmin=0, vmax=8)
axs[1, 0].set_title("(c) Max Velocity (m/s)", fontsize=12, weight="bold")
fig.colorbar(im2, ax=axs[1, 0], shrink=0.7, label="Velocity (m/s)")

# Panel 4: Arrival Time (threshold 0.05 m)
with rasterio.open(post_res["arrival_time_0_05_tif"]) as src:
    arr = src.read(1)
    arr_min = np.ma.masked_invalid(arr) / 60.0
im3 = axs[1, 1].imshow(arr_min, cmap="viridis_r", vmin=0, vmax=90)
axs[1, 1].set_title("(d) Arrival Time at 0.05 m (min) [UNVERIFIED]", fontsize=12, weight="bold")
fig.colorbar(im3, ax=axs[1, 1], shrink=0.7, label="Arrival Time (minutes)")

for ax in axs.flat:
    ax.set_xticks([])
    ax.set_yticks([])

title_txt = (
    "PravahX Model Intercomparison: Tier-0 HAND vs Tier-1 Delft3D FM\n"
    "Ganga Reach near Rishikesh | Hypothetical Scenario (Q_peak = 5,000 m3/s)"
)
plt.suptitle(title_txt, fontsize=14, weight="bold")
plt.tight_layout()

summary_png = out_dir / "ganga_tier1_summary.png"
plt.savefig(summary_png, dpi=180, bbox_inches="tight")
plt.show()

# Save results archive to Drive
ts_str = time.strftime("%Y%m%d_%H%M%S")
drive_res_dir = Path(f"/content/drive/MyDrive/PravahX/ganga_tier1_results_{ts_str}")
drive_res_dir.mkdir(parents=True, exist_ok=True)

for f in out_dir.iterdir():
    shutil.copy(f, drive_res_dir / f.name)
print(f"All results and summary maps saved to Drive: {drive_res_dir}")